In [ ]:
!pip install maldideepkit --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive')
except ImportError:
    print('Running locally -- skipping Google Drive mount')

# Hyperparameter Tuning — MaldiDeepKit Classifiers

This notebook tunes all four MaldiDeepKit architectures (MLP, CNN, ResNet,
Transformer) on the DRIAMS A2018 *S. aureus* + Ciprofloxacin subset,
using the same pipeline as `0-0-1-duroux_analysis`.

## What this notebook covers

1. Load A2018 spectra + metadata, filter to Ciprofloxacin
2. Per-architecture **grid search** over `learning_rate` x `dropout`
3. Evaluate each combination on the **external validation split** (leak-safe)
4. Retrain best config, tune decision threshold, evaluate on test set
5. Compare tuned vs **untuned** (0-0-1 original hardcoded params)

## Grid parameters

```python
LR_GRID   = np.linspace(1e-4, 1e-3, 4)   # -> [1e-4, 4e-4, 7e-4, 1e-3]
DROP_GRID = np.linspace(0.2, 0.6, 4)     # -> [0.2, 0.33, 0.47, 0.6]
```

Tweak `linspace(start, stop, N)` to adjust range/density.

In [ ]:
# =============================================================================
# 1. IMPORTS & CONFIGURATION
# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── MaldiDeepKit classifiers ──
from maldideepkit.attention.mlp import MaldiMLPClassifier
from maldideepkit.cnn.cnn import MaldiCNNClassifier
from maldideepkit.resnet.resnet import MaldiResNetClassifier
from maldideepkit.transformer.transformer import MaldiTransformerClassifier

# ── Metrics ──
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay,
)

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
np.random.seed(SEED)

# ── Paths ──
# Local (uncomment if running on your machine)
DATA_DIR = Path("../../A2018/A2018")
OUT_DIR  = Path("./Tuning_results")

# Colab (uncomment if running on Google Colab)
# DATA_DIR = Path("/content/drive/MyDrive/Flower/Duroux_D_dirams/A2018/A2018")
# OUT_DIR  = Path("/content/drive/MyDrive/Flower/Duroux_D_dirams/0-Analysis/0-0-1-1-Tuning/Tuning_results")

SPECTRA_PATH   = DATA_DIR / "rawSpectra_data.npy"
SPLITS_PATH    = DATA_DIR / "data_splits.csv"
LONG_TABLE_PATH = DATA_DIR / "combined_long_table.csv"

OUT_DIR.mkdir(exist_ok=True)

print(f"Data dir:  {DATA_DIR.resolve()}")
print(f"Output to: {OUT_DIR.resolve()}")

In [ ]:
# Enable interactive 3D plot rotation — requires:  pip install ipympl
# If ipympl is not installed, plots fall back to static inline rendering.
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception:
    pass


---
## 2. LOAD SPECTRA

In [ ]:
X_full = np.load(SPECTRA_PATH).astype("float32")
n_samples, n_bins = X_full.shape
mz_axis = np.arange(2000, 2000 + n_bins * 3, 3)

print(f"Spectra matrix: {X_full.shape}")
print(f"  Samples  {n_samples}")
print(f"  Bins     {n_bins}  (3 Da from {mz_axis[0]} to {mz_axis[-1]} Da)")

---
## 3. LOAD METADATA

In [ ]:
splits_df = pd.read_csv(SPLITS_PATH)
long_df   = pd.read_csv(LONG_TABLE_PATH)

print("Split distribution:")
print(splits_df["Set"].value_counts().to_string())

pivot_df = long_df.pivot_table(index="sample_id", columns="drug", values="response")
pivot_df = pivot_df.reindex(splits_df["sample_id"])

print(f"\nPivot shape: {pivot_df.shape}")
print(f"\nTop 10 drugs by tested samples:")
for drug, n in pivot_df.notna().sum().sort_values(ascending=False).head(10).items():
    print(f"  {drug:20s}  {n:5d}")

---
## 4. PREPARE CIPROFLOXACIN SUBSET

In [ ]:
def prepare_drug_data(drug_name):
    valid_mask = pivot_df[drug_name].notna().to_numpy()
    X = X_full[valid_mask]
    y = pivot_df[drug_name].dropna().to_numpy(dtype=np.int64)
    split_subset = splits_df.loc[valid_mask, "Set"].to_numpy()
    masks = {
        "train": (split_subset == "train"),
        "test":  (split_subset == "test"),
        "valid": (split_subset == "validation"),
    }
    return X, y, masks

DRUG = "Ciprofloxacin"
X_cip, y_cip, masks_cip = prepare_drug_data(DRUG)

# Train for fitting; validation for grid-search scoring; test for final eval.
# Classifier internal val_fraction splits train for early-stopping.
X_train, y_train = X_cip[masks_cip["train"]], y_cip[masks_cip["train"]]
X_valid, y_valid = X_cip[masks_cip["valid"]], y_cip[masks_cip["valid"]]
X_test,  y_test  = X_cip[masks_cip["test"]],  y_cip[masks_cip["test"]]

print(f"Drug: {DRUG}")
print(f"  train  n={len(y_train):5d}  R={(y_train==0).sum():5d}  S={(y_train==1).sum():5d}")
print(f"  valid  n={len(y_valid):5d}  R={(y_valid==0).sum():5d}  S={(y_valid==1).sum():5d}")
print(f"  test   n={len(y_test):5d}  R={(y_test==0).sum():5d}  S={(y_test==1).sum():5d}")

# ── Parametrisable grids ──
# Tweak these to adjust the search space:
LR_GRID   = np.linspace(1e-4, 1e-3, 4)    # learning rate
DROP_GRID = np.linspace(0.2,  0.6,  4)    # dropout (scalar, or high for MLP)
THRESHOLDS = np.linspace(0.05, 0.95, 91)

print(f"\nLR grid:   {LR_GRID}")
print(f"Drop grid:  {DROP_GRID}")

---
## 5. MLP TUNING

In [ ]:
print("=" * 60)
print("MLP — grid search over lr x dropout")
print("=" * 60)

mlp_results = []
mlp_best_balacc = -1
mlp_best_lr = None
mlp_best_dh = None
mlp_best_dl = None

for lr in LR_GRID:
    for d in DROP_GRID:
        dh, dl = d, d / 2
        clf = MaldiMLPClassifier(
            hidden_dim=512, head_dims=(256, 128),
            use_attention=False,
            dropout_high=dh, dropout_low=dl,
            learning_rate=lr,
            weight_decay=1e-3,
            epochs=50, batch_size=64,
            warmup_epochs=10,
            val_fraction=0.1,
            input_transform="log1p+standardize",
            tune_threshold=False,
            random_state=SEED,
            verbose=False,
        )
        clf.fit(X_train, y_train)

        proba_val = clf.predict_proba(X_valid)[:, 1]
        preds_val = clf.predict(X_valid)
        val_balacc = balanced_accuracy_score(y_valid, preds_val)
        val_auc = roc_auc_score(y_valid, proba_val)

        mlp_results.append({"lr": lr, "dh": dh, "dl": dl,
                           "val_balacc": val_balacc, "val_auc": val_auc})
        mark = " *" if val_balacc > mlp_best_balacc else ""
        print(f"  lr={lr:.0e}  drop=({dh:.2f},{dl:.2f})  "
              f"val_balacc={val_balacc:.4f}  val_auc={val_auc:.4f}{mark}")

        if val_balacc > mlp_best_balacc:
            mlp_best_balacc = val_balacc
            mlp_best_lr = lr
            mlp_best_dh = dh
            mlp_best_dl = dl

print(f"\nBest MLP: lr={mlp_best_lr}, drop=({mlp_best_dh:.2f},{mlp_best_dl:.2f})  "
      f"val_balacc={mlp_best_balacc:.4f}")

# ── Plot grid results (3D trisurf) ──
from mpl_toolkits.mplot3d import Axes3D  # noqa: F811

# Prepare data arrays from results list
lrs_log = np.log10([r["lr"] for r in mlp_results])
drops   = [r["dh"] for r in mlp_results]
bals    = [r["val_balacc"] for r in mlp_results]
aucs    = [r["val_auc"] for r in mlp_results]

# Locate best-combo AUC (best is chosen by BalAcc, not AUC)
auc_at_best = [r["val_auc"] for r in mlp_results
               if abs(r["lr"] - mlp_best_lr) < 1e-7 and abs(r["dh"] - mlp_best_dh) < 1e-7][0]

fig = plt.figure(figsize=(14, 6))

# ── Left subplot: Val Balanced Accuracy ──
ax1 = fig.add_subplot(121, projection="3d")
trisurf1 = ax1.plot_trisurf(lrs_log, drops, bals, cmap="viridis", alpha=0.85)
ax1.scatter(np.log10(mlp_best_lr), mlp_best_dh, mlp_best_balacc,
            color="red", s=80, marker="*", edgecolors="k", linewidth=0.5)
ax1.set_xlabel("Learning Rate (log10)")
ax1.set_ylabel("Dropout (high)")
ax1.set_zlabel("Val Balanced Accuracy")
ax1.set_title(f"MLP Tuning — Val BalAcc ({DRUG})")
ax1.set_xticks(np.log10(LR_GRID))
ax1.set_xticklabels([f"{{lr:.0e}}" for lr in LR_GRID], fontsize=7)
fig.colorbar(trisurf1, ax=ax1, shrink=0.5, aspect=10, pad=0.1)

# ── Right subplot: Val AUC ──
ax2 = fig.add_subplot(122, projection="3d")
trisurf2 = ax2.plot_trisurf(lrs_log, drops, aucs, cmap="plasma", alpha=0.85)
ax2.scatter(np.log10(mlp_best_lr), mlp_best_dh, auc_at_best,
            color="red", s=80, marker="*", edgecolors="k", linewidth=0.5)
ax2.set_xlabel("Learning Rate (log10)")
ax2.set_ylabel("Dropout (high)")
ax2.set_zlabel("Val AUC")
ax2.set_title(f"MLP Tuning — Val AUC ({DRUG})")
ax2.set_xticks(np.log10(LR_GRID))
ax2.set_xticklabels([f"{{lr:.0e}}" for lr in LR_GRID], fontsize=7)
fig.colorbar(trisurf2, ax=ax2, shrink=0.5, aspect=10, pad=0.1)

plt.tight_layout()
plt.savefig(OUT_DIR / "grid_search_mlp.pdf", bbox_inches="tight")
plt.show()

# ──────────────────────────────────────────────────────────────────────────────
# Stage 2: Fine grid around best coarse params
# ──────────────────────────────────────────────────────────────────────────────
# Zoom in log-symmetrically for LR (±2×) and linearly for dropout (±0.12 window).
# If a better combo is found, mlp_best_lr / mlp_best_dh / mlp_best_balacc are updated
# so the retrain cell (cell 18) uses the overall optimum from both stages.

LR_FACTOR = 2.0
DROP_WINDOW = 0.12

LR_GRID_FINE = np.logspace(
    np.log10(max(1e-6, mlp_best_lr / LR_FACTOR)),
    np.log10(mlp_best_lr * LR_FACTOR),
    4
)
DROP_GRID_FINE = np.linspace(
    max(0.05, mlp_best_dh - DROP_WINDOW),
    min(0.95, mlp_best_dh + DROP_WINDOW),
    4
)

print(f"\nFine grid around coarse best: lr={mlp_best_lr:.2e}, drop=({mlp_best_dh:.2f},{mlp_best_dl:.2f})")
print(f"  LR_GRID_FINE:   {[f'{lr:.2e}' for lr in LR_GRID_FINE]}")
print(f"  DROP_GRID_FINE: {[f'{d:.3f}' for d in DROP_GRID_FINE]}")
print()

mlp_results_fine = []
for lr in LR_GRID_FINE:
    for d in DROP_GRID_FINE:
        dh, dl = d, d / 2
        mlp_tmp = MaldiMLPClassifier(
            hidden_dim=512,
            head_dims=(256, 128),
            use_attention=False,
            dropout_high=dh,
            dropout_low=dl,
            weight_decay=1e-3,
            learning_rate=lr,
            batch_size=64,
            epochs=50,
            early_stopping_patience=10,
            warmup_epochs=10,
            val_fraction=0.1,
            use_sam=False,
            input_transform="log1p+standardize",
            tune_threshold=False,
            random_state=SEED,
            verbose=False,
        )
        mlp_tmp.fit(X_train, y_train)

        proba_val = mlp_tmp.predict_proba(X_valid)[:, 1]
        preds_val = mlp_tmp.predict(X_valid)
        val_balacc = balanced_accuracy_score(y_valid, preds_val)
        val_auc = roc_auc_score(y_valid, proba_val)

        mlp_results_fine.append({
            "lr": lr, "dh": dh, "dl": dl,
            "val_balacc": val_balacc, "val_auc": val_auc,
        })

        mark = " *" if val_balacc > mlp_best_balacc else ""
        print(f"  lr={lr:.2e}  drop=({dh:.3f},{dl:.3f})  "
              f"val_balacc={val_balacc:.4f}  val_auc={val_auc:.4f}{mark}")

        if val_balacc > mlp_best_balacc:
            mlp_best_balacc = val_balacc
            mlp_best_lr = lr
            mlp_best_dh = dh
            mlp_best_dl = dl

print(f"\nOverall best after fine search: lr={mlp_best_lr:.2e}, "
      f"drop=({mlp_best_dh:.3f},{mlp_best_dl:.3f})  val_balacc={mlp_best_balacc:.4f}")

# ── Plot fine grid results (3D trisurf) ──
from mpl_toolkits.mplot3d import Axes3D  # noqa: F811

lrs_log_fine = np.log10([r["lr"] for r in mlp_results_fine])
drops_fine   = [r["dh"] for r in mlp_results_fine]
bals_fine    = [r["val_balacc"] for r in mlp_results_fine]
aucs_fine    = [r["val_auc"] for r in mlp_results_fine]

auc_at_best_fine = [r["val_auc"] for r in mlp_results_fine
                     if abs(r["lr"] - mlp_best_lr) < 1e-7 and abs(r["dh"] - mlp_best_dh) < 1e-7][0]

fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection="3d")
trisurf1 = ax1.plot_trisurf(lrs_log_fine, drops_fine, bals_fine, cmap="viridis", alpha=0.85)
ax1.scatter(np.log10(mlp_best_lr), mlp_best_dh, mlp_best_balacc,
            color="red", s=80, marker="*", edgecolors="k", linewidth=0.5)
ax1.set_xlabel("Learning Rate (log10)")
ax1.set_ylabel("Dropout (high)")
ax1.set_zlabel("Val Balanced Accuracy")
ax1.set_title(f"Fine Grid — Val BalAcc ({{DRUG}})")
ax1.set_xticks(np.log10(LR_GRID_FINE))
ax1.set_xticklabels([f"${{lr:.2e}}$" for lr in LR_GRID_FINE], fontsize=7)
fig.colorbar(trisurf1, ax=ax1, shrink=0.5, aspect=10, pad=0.1)

ax2 = fig.add_subplot(122, projection="3d")
trisurf2 = ax2.plot_trisurf(lrs_log_fine, drops_fine, aucs_fine, cmap="plasma", alpha=0.85)
ax2.scatter(np.log10(mlp_best_lr), mlp_best_dh, auc_at_best_fine,
            color="red", s=80, marker="*", edgecolors="k", linewidth=0.5)
ax2.set_xlabel("Learning Rate (log10)")
ax2.set_ylabel("Dropout (high)")
ax2.set_zlabel("Val AUC")
ax2.set_title(f"Fine Grid — Val AUC ({{DRUG}})")
ax2.set_xticks(np.log10(LR_GRID_FINE))
ax2.set_xticklabels([f"${{lr:.2e}}$" for lr in LR_GRID_FINE], fontsize=7)
fig.colorbar(trisurf2, ax=ax2, shrink=0.5, aspect=10, pad=0.1)

plt.tight_layout()
plt.savefig(OUT_DIR / "grid_search_mlp_fine.pdf", bbox_inches="tight")
plt.show()
# ── Retrain best + threshold tuning ──
mlp_tuned = MaldiMLPClassifier(
    hidden_dim=512, head_dims=(256,128), use_attention=False,
    dropout_high=mlp_best_dh, dropout_low=mlp_best_dl,
    learning_rate=mlp_best_lr, weight_decay=1e-3,
    epochs=50, batch_size=64, warmup_epochs=10,
    val_fraction=0.1,
    input_transform="log1p+standardize",
    tune_threshold=False, random_state=SEED, verbose=True,
)
mlp_tuned.fit(X_train, y_train)

# Evaluation
mlp_tuned_metrics = {}
for name, X_s, y_s in [("train",X_train,y_train),("test",X_test,y_test),("valid",X_valid,y_valid)]:
    preds = mlp_tuned.predict(X_s)
    proba = mlp_tuned.predict_proba(X_s)[:, 1]
    mlp_tuned_metrics[name] = {
        "BalAcc": balanced_accuracy_score(y_s, preds),
        "AUC":    roc_auc_score(y_s, proba),
    }

# Threshold tuning on validation
proba_val = mlp_tuned.predict_proba(X_valid)[:, 1]
balaccs_v = [balanced_accuracy_score(y_valid, proba_val >= t) for t in THRESHOLDS]
mlp_best_thresh = THRESHOLDS[np.argmax(balaccs_v)]
mlp_test_preds = (mlp_tuned.predict_proba(X_test)[:, 1] >= mlp_best_thresh)
mlp_test_balacc = balanced_accuracy_score(y_test, mlp_test_preds)
mlp_test_auc    = roc_auc_score(y_test, mlp_tuned.predict_proba(X_test)[:, 1])

print(f"\nMLP Tuned — test BalAcc={mlp_test_balacc:.4f}  test AUC={mlp_test_auc:.4f}  (thresh={mlp_best_thresh:.3f})")

# 0-0-1 original untuned reference (run notebook to get actual values)
mlp_untuned_balacc = 0.5886
mlp_untuned_auc    = 0.6877

---
## 6. CNN TUNING

In [ ]:
print("=" * 60)
print("CNN — grid search over lr x dropout")
print("=" * 60)

cnn_results = []
cnn_best_balacc = -1
cnn_best_lr = None
cnn_best_d = None

for lr in LR_GRID:
    for d in DROP_GRID:
        clf = MaldiCNNClassifier(
            dropout=d,
            learning_rate=lr,
            epochs=50, batch_size=64,
            val_fraction=0.1,
            input_transform="log1p+standardize",
            tune_threshold=False,
            random_state=SEED,
            verbose=False,
        )
        clf.fit(X_train, y_train)

        proba_val = clf.predict_proba(X_valid)[:, 1]
        preds_val = clf.predict(X_valid)
        val_balacc = balanced_accuracy_score(y_valid, preds_val)
        val_auc = roc_auc_score(y_valid, proba_val)

        cnn_results.append({"lr": lr, "d": d, "val_balacc": val_balacc, "val_auc": val_auc})
        mark = " *" if val_balacc > cnn_best_balacc else ""
        print(f"  lr={lr:.0e}  drop={d:.2f}  "
              f"val_balacc={val_balacc:.4f}  val_auc={val_auc:.4f}{mark}")

        if val_balacc > cnn_best_balacc:
            cnn_best_balacc = val_balacc
            cnn_best_lr = lr
            cnn_best_d = d

print(f"\nBest CNN: lr={cnn_best_lr}, drop={cnn_best_d:.2f}  val_balacc={cnn_best_balacc:.4f}")

# ── Plot ──
fig, ax = plt.subplots(figsize=(8, 5))
for j, d in enumerate(DROP_GRID):
    subset = [r for r in cnn_results if abs(r["d"] - d) < 0.001]
    lrs = [r["lr"] for r in subset]
    bals = [r["val_balacc"] for r in subset]
    order = np.argsort(lrs)
    ax.semilogx(np.array(lrs)[order], np.array(bals)[order], marker="o",
                label=f"drop={d:.2f}")
ax.set_xlabel("Learning Rate (log scale)")
ax.set_ylabel("Val Balanced Accuracy")
ax.set_title(f"CNN Tuning — {DRUG}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "grid_search_cnn.pdf", bbox_inches="tight")
plt.show()

# ── Retrain best + threshold tuning ──
cnn_tuned = MaldiCNNClassifier(
    dropout=cnn_best_d,
    learning_rate=cnn_best_lr,
    epochs=50, batch_size=64,
    val_fraction=0.1,
    input_transform="log1p+standardize",
    tune_threshold=False, random_state=SEED, verbose=True,
)
cnn_tuned.fit(X_train, y_train)

# Threshold tuning
proba_val = cnn_tuned.predict_proba(X_valid)[:, 1]
balaccs_v = [balanced_accuracy_score(y_valid, proba_val >= t) for t in THRESHOLDS]
cnn_best_thresh = THRESHOLDS[np.argmax(balaccs_v)]
cnn_test_preds = (cnn_tuned.predict_proba(X_test)[:, 1] >= cnn_best_thresh)
cnn_test_balacc = balanced_accuracy_score(y_test, cnn_test_preds)
cnn_test_auc = roc_auc_score(y_test, cnn_tuned.predict_proba(X_test)[:, 1])

print(f"\nCNN Tuned — test BalAcc={cnn_test_balacc:.4f}  test AUC={cnn_test_auc:.4f}  (thresh={cnn_best_thresh:.3f})")

cnn_untuned_balacc = 0.58
cnn_untuned_auc    = 0.70

---
## 7. RESNET TUNING

In [ ]:
print("=" * 60)
print("ResNet — grid search over lr x dropout")
print("=" * 60)

resnet_results = []
resnet_best_balacc = -1
resnet_best_lr = None
resnet_best_d = None

for lr in LR_GRID:
    for d in DROP_GRID:
        clf = MaldiResNetClassifier(
            dropout=d,
            learning_rate=lr,
            epochs=50, batch_size=64,
            warmup_epochs=5,
            val_fraction=0.1,
            input_transform="log1p",
            tune_threshold=False,
            random_state=SEED,
            verbose=False,
        )
        clf.fit(X_train, y_train)

        proba_val = clf.predict_proba(X_valid)[:, 1]
        preds_val = clf.predict(X_valid)
        val_balacc = balanced_accuracy_score(y_valid, preds_val)
        val_auc = roc_auc_score(y_valid, proba_val)

        resnet_results.append({"lr": lr, "d": d, "val_balacc": val_balacc, "val_auc": val_auc})
        mark = " *" if val_balacc > resnet_best_balacc else ""
        print(f"  lr={lr:.0e}  drop={d:.2f}  "
              f"val_balacc={val_balacc:.4f}  val_auc={val_auc:.4f}{mark}")

        if val_balacc > resnet_best_balacc:
            resnet_best_balacc = val_balacc
            resnet_best_lr = lr
            resnet_best_d = d

print(f"\nBest ResNet: lr={resnet_best_lr}, drop={resnet_best_d:.2f}  val_balacc={resnet_best_balacc:.4f}")

# ── Plot ──
fig, ax = plt.subplots(figsize=(8, 5))
for j, d in enumerate(DROP_GRID):
    subset = [r for r in resnet_results if abs(r["d"] - d) < 0.001]
    lrs = [r["lr"] for r in subset]
    bals = [r["val_balacc"] for r in subset]
    order = np.argsort(lrs)
    ax.semilogx(np.array(lrs)[order], np.array(bals)[order], marker="o",
                label=f"drop={d:.2f}")
ax.set_xlabel("Learning Rate (log scale)")
ax.set_ylabel("Val Balanced Accuracy")
ax.set_title(f"ResNet Tuning — {DRUG}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "grid_search_resnet.pdf", bbox_inches="tight")
plt.show()

# ── Retrain best + threshold tuning ──
resnet_tuned = MaldiResNetClassifier(
    dropout=resnet_best_d,
    learning_rate=resnet_best_lr,
    epochs=50, batch_size=64,
    warmup_epochs=5,
    val_fraction=0.1,
    input_transform="log1p",
    tune_threshold=False, random_state=SEED, verbose=True,
)
resnet_tuned.fit(X_train, y_train)

# Threshold tuning
proba_val = resnet_tuned.predict_proba(X_valid)[:, 1]
balaccs_v = [balanced_accuracy_score(y_valid, proba_val >= t) for t in THRESHOLDS]
resnet_best_thresh = THRESHOLDS[np.argmax(balaccs_v)]
resnet_test_preds = (resnet_tuned.predict_proba(X_test)[:, 1] >= resnet_best_thresh)
resnet_test_balacc = balanced_accuracy_score(y_test, resnet_test_preds)
resnet_test_auc = roc_auc_score(y_test, resnet_tuned.predict_proba(X_test)[:, 1])

print(f"\nResNet Tuned — test BalAcc={resnet_test_balacc:.4f}  test AUC={resnet_test_auc:.4f}  (thresh={resnet_best_thresh:.3f})")

resnet_untuned_balacc = 0.57
resnet_untuned_auc    = 0.69

---
## 8. TRANSFORMER TUNING

In [ ]:
print("=" * 60)
print("Transformer — grid search over lr x dropout")
print("=" * 60)

transformer_results = []
transformer_best_balacc = -1
transformer_best_lr = None
transformer_best_d = None

for lr in LR_GRID:
    for d in DROP_GRID:
        clf = MaldiTransformerClassifier(
            dropout=d,
            learning_rate=lr,
            epochs=50, batch_size=64,
            val_fraction=0.1,
            input_transform="log1p+standardize",
            tune_threshold=False,
            random_state=SEED,
            verbose=False,
        )
        clf.fit(X_train, y_train)

        proba_val = clf.predict_proba(X_valid)[:, 1]
        preds_val = clf.predict(X_valid)
        val_balacc = balanced_accuracy_score(y_valid, preds_val)
        val_auc = roc_auc_score(y_valid, proba_val)

        transformer_results.append({"lr": lr, "d": d, "val_balacc": val_balacc, "val_auc": val_auc})
        mark = " *" if val_balacc > transformer_best_balacc else ""
        print(f"  lr={lr:.0e}  drop={d:.2f}  "
              f"val_balacc={val_balacc:.4f}  val_auc={val_auc:.4f}{mark}")

        if val_balacc > transformer_best_balacc:
            transformer_best_balacc = val_balacc
            transformer_best_lr = lr
            transformer_best_d = d

print(f"\nBest Transformer: lr={transformer_best_lr}, drop={transformer_best_d:.2f}  val_balacc={transformer_best_balacc:.4f}")

# ── Plot ──
fig, ax = plt.subplots(figsize=(8, 5))
for j, d in enumerate(DROP_GRID):
    subset = [r for r in transformer_results if abs(r["d"] - d) < 0.001]
    lrs = [r["lr"] for r in subset]
    bals = [r["val_balacc"] for r in subset]
    order = np.argsort(lrs)
    ax.semilogx(np.array(lrs)[order], np.array(bals)[order], marker="o",
                label=f"drop={d:.2f}")
ax.set_xlabel("Learning Rate (log scale)")
ax.set_ylabel("Val Balanced Accuracy")
ax.set_title(f"Transformer Tuning — {DRUG}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "grid_search_transformer.pdf", bbox_inches="tight")
plt.show()

# ── Retrain best + threshold tuning ──
transformer_tuned = MaldiTransformerClassifier(
    dropout=transformer_best_d,
    learning_rate=transformer_best_lr,
    epochs=50, batch_size=64,
    val_fraction=0.1,
    input_transform="log1p+standardize",
    tune_threshold=False, random_state=SEED, verbose=True,
)
transformer_tuned.fit(X_train, y_train)

# Threshold tuning
proba_val = transformer_tuned.predict_proba(X_valid)[:, 1]
balaccs_v = [balanced_accuracy_score(y_valid, proba_val >= t) for t in THRESHOLDS]
transformer_best_thresh = THRESHOLDS[np.argmax(balaccs_v)]
transformer_test_preds = (transformer_tuned.predict_proba(X_test)[:, 1] >= transformer_best_thresh)
transformer_test_balacc = balanced_accuracy_score(y_test, transformer_test_preds)
transformer_test_auc = roc_auc_score(y_test, transformer_tuned.predict_proba(X_test)[:, 1])

print(f"\nTransformer Tuned — test BalAcc={transformer_test_balacc:.4f}  test AUC={transformer_test_auc:.4f}  (thresh={transformer_best_thresh:.3f})")

transformer_untuned_balacc = 0.56
transformer_untuned_auc    = 0.68

---
## 9. COMPARISON — TUNED vs UNTUNED

In [ ]:
summary_df = pd.DataFrame([
    {"Architecture":"MLP",         "Tuned BalAcc":mlp_test_balacc,         "Tuned AUC":mlp_test_auc,
     "Untuned BalAcc":mlp_untuned_balacc, "Untuned AUC":mlp_untuned_auc},
    {"Architecture":"CNN",         "Tuned BalAcc":cnn_test_balacc,         "Tuned AUC":cnn_test_auc,
     "Untuned BalAcc":cnn_untuned_balacc, "Untuned AUC":cnn_untuned_auc},
    {"Architecture":"ResNet",      "Tuned BalAcc":resnet_test_balacc,      "Tuned AUC":resnet_test_auc,
     "Untuned BalAcc":resnet_untuned_balacc, "Untuned AUC":resnet_untuned_auc},
    {"Architecture":"Transformer", "Tuned BalAcc":transformer_test_balacc, "Tuned AUC":transformer_test_auc,
     "Untuned BalAcc":transformer_untuned_balacc, "Untuned AUC":transformer_untuned_auc},
])

summary_df["delta BalAcc"] = summary_df["Tuned BalAcc"] - summary_df["Untuned BalAcc"]
summary_df["delta AUC"]    = summary_df["Tuned AUC"]    - summary_df["Untuned AUC"]

print(summary_df.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(4)
w = 0.2
ax.bar(x - 1.5*w, summary_df["Untuned BalAcc"], w, label="Untuned BalAcc", color="#aec7e8")
ax.bar(x - 0.5*w, summary_df["Tuned BalAcc"],   w, label="Tuned BalAcc",   color="#1f77b4")
ax.bar(x + 0.5*w, summary_df["Untuned AUC"],    w, label="Untuned AUC",    color="#ffbb78")
ax.bar(x + 1.5*w, summary_df["Tuned AUC"],      w, label="Tuned AUC",      color="#ff7f0e")
ax.set_xticks(x)
ax.set_xticklabels(summary_df["Architecture"])
ax.set_ylabel("Score")
ax.set_title(f"Tuned vs Untuned — {DRUG}")
ax.legend(loc="lower right", fontsize=7)
ax.set_ylim(0, 1)
ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.tight_layout()
plt.savefig(OUT_DIR / "tuned_vs_untuned.pdf", bbox_inches="tight")
plt.show()

# ROC overlay (tuned)
fig, ax = plt.subplots(figsize=(7, 7))
for name, clf in [("MLP", mlp_tuned), ("CNN", cnn_tuned),
                   ("ResNet", resnet_tuned), ("Transformer", transformer_tuned)]:
    RocCurveDisplay.from_predictions(y_test, clf.predict_proba(X_test)[:, 1], name=name, ax=ax)
ax.plot([0,1],[0,1],"k--",alpha=0.3)
ax.set_title(f"ROC Curves (Tuned) — {DRUG}")
plt.tight_layout()
plt.savefig(OUT_DIR / "roc_tuned.pdf", bbox_inches="tight")
plt.show()

---
## 10. REPORT

In [ ]:
report_path = Path("report.md")

top_row = summary_df.loc[summary_df["Tuned BalAcc"].idxmax()]

lines = [
    f"# Deep Classifier Hyperparameter Tuning Report",
    "",
    f"**Dataset:** DRIAMS A2018 — *S. aureus* + {DRUG}",
    f"**LR grid:** linspace({LR_GRID[0]:.0e}, {LR_GRID[-1]:.0e}, {len(LR_GRID)})",
    f"**Dropout grid:** linspace({DROP_GRID[0]:.1f}, {DROP_GRID[-1]:.1f}, {len(DROP_GRID)})",
    f"**Epochs:** 50, batch_size=64, val_fraction=0.1",
    "",
    "---",
    "",
    "## Tuned vs Untuned",
    "",
    "| Architecture | Untuned BalAcc | Tuned BalAcc | Delta | Untuned AUC | Tuned AUC | Delta |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for _, r in summary_df.iterrows():
    lines.append(
        f"| {r['Architecture']} | {r['Untuned BalAcc']:.4f} | {r['Tuned BalAcc']:.4f} | "
        f"{r['delta BalAcc']:+.4f} | {r['Untuned AUC']:.4f} | {r['Tuned AUC']:.4f} | "
        f"{r['delta AUC']:+.4f} |"
    )

lines += [
    "",
    f"**Best architecture (tuned):** {top_row['Architecture']}  "
    f"(BalAcc={top_row['Tuned BalAcc']:.4f}, AUC={top_row['Tuned AUC']:.4f})",
    "",
    "---",
    "## Best Hyperparameters",
    "",
    f"- **MLP:** lr={mlp_best_lr}, drop=({mlp_best_dh:.2f},{mlp_best_dl:.2f})",
    f"- **CNN:** lr={cnn_best_lr}, drop={cnn_best_d:.2f}",
    f"- **ResNet:** lr={resnet_best_lr}, drop={resnet_best_d:.2f}",
    f"- **Transformer:** lr={transformer_best_lr}, drop={transformer_best_d:.2f}",
    "",
    "---",
    "## Generated Figures",
    "",
    f"- `{OUT_DIR}/grid_search_mlp.pdf`",
    f"- `{OUT_DIR}/grid_search_cnn.pdf`",
    f"- `{OUT_DIR}/grid_search_resnet.pdf`",
    f"- `{OUT_DIR}/grid_search_transformer.pdf`",
    f"- `{OUT_DIR}/tuned_vs_untuned.pdf`",
    f"- `{OUT_DIR}/roc_tuned.pdf`",
    "",
    f"*Report generated by `0-0-1-1-Tuning.ipynb`*",
]

with open(report_path, "w") as f:
    f.write("\n".join(lines) + "\n")
print(f"Report saved to: {report_path.resolve()}")
print(report_path.read_text())